In [1]:
import sys
print(sys.executable)


/Users/kosiew/GitHub/datafusion-python/.venv/bin/python3


In [2]:
import time
import threading
import pyarrow as pa
from datafusion import SessionContext

In [3]:
def create_large_dataset():
    """Create a large dataset for testing interruption."""
    ctx = SessionContext()
    
    # Create large record batches similar to the test
    batches = []
    for i in range(10):
        batch = pa.RecordBatch.from_arrays(
            [
                pa.array(list(range(i * 1000, (i + 1) * 1000))),
                pa.array([f"value_{j}" for j in range(i * 1000, (i + 1) * 1000)]),
            ],
            names=["a", "b"],
        )
        batches.append(batch)
    
    # Register tables
    ctx.register_record_batches("t1", [batches])
    ctx.register_record_batches("t2", [batches])
    
    return ctx

# Setup the test environment
ctx = create_large_dataset()
print("Dataset created successfully!")

Dataset created successfully!


In [5]:
# Create a complex, long-running query
df = ctx.sql("""
    WITH t1_expanded AS (
        SELECT 
            a, 
            b, 
            CAST(a AS DOUBLE) / 1.5 AS c,
            CAST(a AS DOUBLE) * CAST(a AS DOUBLE) AS d
        FROM t1
        CROSS JOIN (SELECT 1 AS dummy FROM t1 LIMIT 5)
    ),
    t2_expanded AS (
        SELECT 
            a,
            b,
            CAST(a AS DOUBLE) * 2.5 AS e,
            CAST(a AS DOUBLE) * CAST(a AS DOUBLE) * CAST(a AS DOUBLE) AS f
        FROM t2
        CROSS JOIN (SELECT 1 AS dummy FROM t2 LIMIT 5)
    )
    SELECT 
        t1.a, t1.b, t1.c, t1.d, 
        t2.a AS a2, t2.b AS b2, t2.e, t2.f
    FROM t1_expanded t1
    JOIN t2_expanded t2 ON t1.a % 100 = t2.a % 100
    WHERE t1.a > 100 AND t2.a > 100
""")

print("Starting long-running query...")
print("Press Ctrl+C to interrupt!")

try:
    result = df.collect()
    print(f"Query completed successfully! Got {len(result)} batches")
except KeyboardInterrupt:
    print("✅ Query was successfully interrupted by Ctrl+C!")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

Starting long-running query...
Press Ctrl+C to interrupt!
Query completed successfully! Got 2996 batches


In [6]:
df = ctx.sql("""
    WITH t1_expanded AS (
        SELECT 
            a, 
            b, 
            CAST(a AS DOUBLE) / 1.5 AS c,
            CAST(a AS DOUBLE) * CAST(a AS DOUBLE) AS d
        FROM t1
        CROSS JOIN (SELECT 1 AS dummy FROM t1 LIMIT 5)
    ),
    t2_expanded AS (
        SELECT 
            a,
            b,
            CAST(a AS DOUBLE) * 2.5 AS e,
            CAST(a AS DOUBLE) * CAST(a AS DOUBLE) * CAST(a AS DOUBLE) AS f
        FROM t2
        CROSS JOIN (SELECT 1 AS dummy FROM t2 LIMIT 5)
    )
    SELECT 
        t1.a, t1.b, t1.c, t1.d, 
        t2.a AS a2, t2.b AS b2, t2.e, t2.f
    FROM t1_expanded t1
    JOIN t2_expanded t2 ON t1.a % 100 = t2.a % 100
    WHERE t1.a > 100 AND t2.a > 100
""")

In [7]:
ctx = create_very_large_dataset()

NameError: name 'create_very_large_dataset' is not defined

In [8]:
def create_very_large_dataset():
    """Create a much larger dataset that will take time to process."""
    ctx = SessionContext()
    
    # Create much larger record batches
    batches = []
    for i in range(100):  # Increased from 10 to 100
        batch = pa.RecordBatch.from_arrays(
            [
                pa.array(list(range(i * 10000, (i + 1) * 10000))),  # 10k rows per batch
                pa.array([f"value_{j}" for j in range(i * 10000, (i + 1) * 10000)]),
                pa.array([j * 1.5 for j in range(i * 10000, (i + 1) * 10000)]),  # Float column
                pa.array([f"category_{j % 1000}" for j in range(i * 10000, (i + 1) * 10000)]),  # Categories
            ],
            names=["id", "text_col", "float_col", "category"],
        )
        batches.append(batch)
    
    # Register multiple large tables
    ctx.register_record_batches("large_table1", batches)
    ctx.register_record_batches("large_table2", batches)
    ctx.register_record_batches("large_table3", batches)
    
    print(f"Created dataset with {len(batches)} batches, ~{len(batches) * 10000:,} rows each")
    return ctx

# Setup the test environment
ctx = create_very_large_dataset()

TypeError: argument 'partitions': 'RecordBatch' object cannot be converted to 'PyList'

In [9]:
import time
import pyarrow as pa
from datafusion import SessionContext

def create_very_large_dataset():
    """Create a much larger dataset that will take time to process."""
    ctx = SessionContext()
    
    # Create much larger record batches
    batches = []
    for i in range(100):  # Increased from 10 to 100
        batch = pa.RecordBatch.from_arrays(
            [
                pa.array(list(range(i * 10000, (i + 1) * 10000))),  # 10k rows per batch
                pa.array([f"value_{j}" for j in range(i * 10000, (i + 1) * 10000)]),
                pa.array([j * 1.5 for j in range(i * 10000, (i + 1) * 10000)]),  # Float column
                pa.array([f"category_{j % 1000}" for j in range(i * 10000, (i + 1) * 10000)]),  # Categories
            ],
            names=["id", "text_col", "float_col", "category"],
        )
        batches.append(batch)
    
    # Fix: Register multiple large tables - wrap batches in a list for partitions
    ctx.register_record_batches("large_table1", [batches])  # List of partitions
    ctx.register_record_batches("large_table2", [batches])  # List of partitions
    ctx.register_record_batches("large_table3", [batches])  # List of partitions
    
    print(f"Created dataset with {len(batches)} batches, ~{len(batches) * 10000:,} rows each")
    return ctx

# Setup the test environment
ctx = create_very_large_dataset()

Created dataset with 100 batches, ~1,000,000 rows each


In [14]:
# This will definitely be slow enough to interrupt
df = ctx.sql("""
    SELECT 
        t1.id,
        t2.id as id2,
        t1.float_col * t2.float_col as product,
        CONCAT(t1.text_col, '_', t2.text_col) as combined_text,
        SIN(t1.float_col) + COS(t2.float_col) as trig_calc,
        CASE 
            WHEN t1.id % 2 = 0 THEN 'even'
            ELSE 'odd'
        END as parity
    FROM large_table1 t1
    CROSS JOIN large_table2 t2
    WHERE t1.id BETWEEN 1000 AND 5000
      AND t2.id BETWEEN 1500 AND 5500
    ORDER BY product DESC
    LIMIT 900000
""")

print("Starting cartesian product query...")
print("Press Ctrl+C to interrupt!")

try:
    result = df.collect()
    print(f"Query completed! Got {len(result)} batches")
except KeyboardInterrupt:
    print("✅ Query was successfully interrupted by Ctrl+C!")
except Exception as e:
    print(f"Error: {e}")

Starting cartesian product query...
Press Ctrl+C to interrupt!
Query completed! Got 110 batches
